# Deploy and Share DustyLM

> **Optional:** This notebook is not part of the core training path. Use it when you want to run a model in the browser or publish your own work.

This notebook covers two practical tasks:

1. **Run a trained model in the browser.** DustyLM training saves PyTorch `.pt` weights, while the browser UI uses `onnxruntime-web`. You will convert the checkpoint to a smaller, quantized ONNX file and optionally serve it locally.
2. **Publish your work to Hugging Face.** If you trained a new checkpoint, created a new persona, or generated a custom dataset, the optional publishing sections show how to stage and upload it to repositories under your own Hugging Face account.

The browser workflow comes first. Model and dataset publishing are separate optional workflows.

## Setup

This notebook packages an existing model, so the default Colab CPU runtime is sufficient.

**Google Colab:** Run the Setup cell below, then continue through the notebook.

**RunPod:** A CPU or GPU pod works for this workflow. Run the Setup cell below.

**Local laptop:**

1. If needed, [install uv](https://docs.astral.sh/uv/getting-started/installation/), then run `uv sync --extra onnx --extra hub` from the repository root.
2. Select the project's `.venv` as your notebook kernel.
3. Run the Setup cell below.

In [ ]:
import os
import sys
from pathlib import Path

is_cloud = "google.colab" in sys.modules or "RUNPOD_POD_ID" in os.environ
if is_cloud:
    !pip install -q uv
    if not Path("Makefile").exists():
        !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
        os.chdir("dusty-lm")
    !pip install -q -e .
elif not Path("Makefile").exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        raise FileNotFoundError("Open this notebook from inside the dusty-lm repository.")

REPO_ROOT = Path.cwd()
print(f"✅ Setup complete. Repo root: {REPO_ROOT}")

<details>
<summary>Windows users</summary>

This notebook uses `make` to keep commands short. The recommended Windows path is WSL, where the cells work as written. On native Windows, install GNU Make with Chocolatey or Scoop before running the command cells.

</details>

## 1. Choose the Model to Deploy

Conversion needs two source files: the trained checkpoint weights and the matching tokenizer. If you completed the [Train from Scratch notebook](02_train_from_scratch.ipynb), the next cell uses your promoted `sft_dusty8m` files without replacing them. In a fresh Colab runtime, it downloads the published Dusty model instead.

Change `EXPORT_PROFILE` only when you have configured another deployable profile.

In [ ]:
from pathlib import Path

from dustylm.config import get_profile

EXPORT_PROFILE = "sft_dusty8m"
profile = get_profile(EXPORT_PROFILE)
required_model_files = [
    Path(profile.generation.checkpoint_path),
    Path(profile.model.tokenizer.path_or_name),
]

missing = [path for path in required_model_files if not path.is_file()]
if missing:
    print("Required model files are missing. Downloading the published Dusty model...")
    !make download-models
else:
    print("Using your existing trained or downloaded model files.")

for path in required_model_files:
    if not path.is_file():
        raise FileNotFoundError(f"Required model file not found: {path}")
    print(f"✅ {path}")

## 2. Convert the Model to ONNX

The browser cannot load the PyTorch `.pt` checkpoint directly. `make export-onnx` converts the selected checkpoint to `docs/model.onnx`, applies dynamic int8 quantization to reduce its size, and copies the matching tokenizer to `docs/tokenizer.json`.

The command also runs the exported graph through ONNX Runtime before reporting success. The browser UI then uses the same graph through `onnxruntime-web`.

In [ ]:
!make export-onnx ONNX_PROFILE={EXPORT_PROFILE}

In [ ]:
onnx_artifacts = [Path("docs/model.onnx"), Path("docs/tokenizer.json")]
for path in onnx_artifacts:
    if not path.is_file():
        raise FileNotFoundError(f"Export did not create: {path}")
    print(f"✅ {path}: {path.stat().st_size / 1_000_000:.2f} MB")

The browser files are now ready. `model.onnx` contains the model graph and quantized weights; `tokenizer.json` converts text to the token IDs expected by that model.

## 3. Try the Browser UI

Choose the path that matches where you are running this notebook.

### Google Colab or Quick Preview

Open the published [DustyLM browser demo](https://khordoo.github.io/dusty-lm/). It uses the published Dusty model, so no local setup is required.

### Your Model on a Local Laptop

1. Uncomment and run the command below.
2. Open `http://localhost:8000`. To use another port, set `WEB_PORT`, as shown below.
3. Chat with the model you converted in section 2.

In [ ]:
# Default: open http://localhost:8000
# !make serve-web

# Different port: open http://localhost:9000
# !make serve-web WEB_PORT=9000

<a href="https://khordoo.github.io/dusty-lm/">
  <img src="../docs/images/dusty-web-ui.png" alt="DustyLM browser interface" width="700">
</a>

## 4. Publish Your Model to Hugging Face (Optional)

Use this section only when you want to share a checkpoint you trained or customized. If you are only using the published Dusty model, you can skip it.

First create a model repository under **your own Hugging Face account**. In the next cell, replace `your-username` with your Hugging Face username. `HF_MODEL_REPO_ID` is the exact upload destination.

Always stage before publishing. `make stage-hub` rebuilds `artifacts/hub_upload/sft_dusty8m/` and stops without uploading, so you can inspect the checkpoint, ONNX model, tokenizer files, model card, and logo.

In [ ]:
# Replace your-username before publishing.
HF_MODEL_REPO_ID = "your-username/dusty-8m-sft"

!make stage-hub HF_REPO_ID={HF_MODEL_REPO_ID}

In [ ]:
model_staging_dir = Path("artifacts/hub_upload/sft_dusty8m")
staged_model_files = sorted(
    path.relative_to(model_staging_dir)
    for path in model_staging_dir.rglob("*")
    if path.is_file()
)
if not staged_model_files:
    raise FileNotFoundError(f"No staged files found in {model_staging_dir}")

print(f"Staged {len(staged_model_files)} files:")
for path in staged_model_files:
    print(f"  {path}")

### 4.1 Authenticate and Publish

Choose one authentication method:

1. Run `hf auth login` and enter a Hugging Face token with write access.
2. Or set the `HF_TOKEN` environment variable to a token with write access.

You do not need both. Confirm that `HF_MODEL_REPO_ID` contains your username and review every staged file before publishing. The commands remain commented so **Run all** cannot upload anything.

In [ ]:
# Option 1: Authenticate interactively once.
# !hf auth login

# Publish only after replacing your-username and reviewing the staged folder.
# !make push-hub HF_REPO_ID={HF_MODEL_REPO_ID}

## 5. Publish Your Dataset to Hugging Face (Optional)

Use this section only when you generated or edited an SFT dataset that you want to share. Model and dataset repositories are separate on Hugging Face.

First create a dataset repository under **your own Hugging Face account**. Replace `your-username` in `HF_DATASET_REPO_ID`; that variable is the exact upload destination.

`make stage-dataset` converts the simplified SFT JSONL to standard conversational `messages`, creates a stratified train/test split, validates the dataset card, and stops without uploading.

In [ ]:
# Replace your-username before publishing.
HF_DATASET_REPO_ID = "your-username/dusty-chat"
HF_DATASET_INPUT = Path("artifacts/datasets/dusty_sft.jsonl")
HF_DATASET_TEST_SIZE = 1500  # Reduce this for a smaller custom dataset.

if HF_DATASET_INPUT.is_file():
    !make stage-dataset HF_DATASET_REPO_ID={HF_DATASET_REPO_ID} HF_DATASET_INPUT={HF_DATASET_INPUT} HF_DATASET_TEST_SIZE={HF_DATASET_TEST_SIZE}
else:
    print(f"Dataset not found: {HF_DATASET_INPUT}")
    print("Complete the dataset steps in Notebook 02 or Notebook 03 before staging it.")

### 5.1 Authenticate and Publish

Use the same authentication choice as model publishing: run `hf auth login`, or set `HF_TOKEN`. Confirm that `HF_DATASET_REPO_ID` contains your username and review `artifacts/hf/HF_DATASET_CARD.md` before running the commented upload command.

In [ ]:
# Publish only after replacing your-username and completing the dry run.
# !make push-dataset HF_DATASET_REPO_ID={HF_DATASET_REPO_ID} HF_DATASET_INPUT={HF_DATASET_INPUT} HF_DATASET_TEST_SIZE={HF_DATASET_TEST_SIZE}